# 03. Feature engineering

Construye variables históricas, ventanas móviles, estadísticas por bus, variables temporales y targets para 7, 5 y 3 días.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
PLOTS_DIR = PROJECT_ROOT / "outputs" / "plots"
METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"

for directory in [DATA_PROCESSED_DIR, MODELS_DIR, PLOTS_DIR, METRICS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


In [2]:
import pandas as pd
from IPython.display import display

from src.feature_engineering import (
    DEFAULT_FEATURE_COLUMNS,
    create_future_targets,
    create_temporal_features,
    generate_cause_based_features,
    generate_inventory_features,
    generate_rolling_features,
    generate_system_features,
    generate_text_pattern_features,
    summarize_feature_quality,
)


In [3]:
eventos_df = pd.read_parquet(DATA_PROCESSED_DIR / "eventos.parquet")
features_df = generate_rolling_features(eventos_df)
features_df = generate_cause_based_features(features_df)
features_df = generate_system_features(features_df)
features_df = generate_inventory_features(features_df)
features_df = generate_text_pattern_features(features_df)
features_df = create_temporal_features(features_df)
features_df = create_future_targets(features_df, windows=(7, 5, 3, 10, 14, 30))
quality_df = summarize_feature_quality(features_df, target_windows=(7, 5, 3, 10, 14, 30))
features_path = DATA_PROCESSED_DIR / "features.parquet"
features_df.to_parquet(features_path, index=False)

print("Filas en features:", len(features_df))
print("Columnas totales en features:", features_df.shape[1])
print("Columnas de modelado:")
print(DEFAULT_FEATURE_COLUMNS)
print("Parquet guardado en:", features_path)
display(quality_df)

display(
    features_df[
        [
            "placa_patente",
            "fecha_evento",
            "dias_desde_correctivo_anterior",
            "correctivos_previos",
            "correctivos_ult_7d",
            "dias_desde_ultima_misma_causa",
            "racha_misma_causa",
            "repuestos_count_evento",
            "count_keyword_freno_ult_7d",
            "correctivos_ult_5d",
            "correctivos_ult_3d",
            "mes",
            "dia_semana",
            "fin_mes",
            "correctivo_prox_7d",
            "correctivo_prox_5d",
            "correctivo_prox_3d",
            "correctivo_prox_10d",
            "correctivo_prox_14d",
            "correctivo_prox_30d",
        ]
    ].head(20)
)


Filas en features: 12207
Columnas de modelado:
['dias_desde_correctivo_anterior', 'correctivos_previos', 'correctivos_ult_7d', 'correctivos_ult_5d', 'correctivos_ult_3d', 'dias_desde_correctivo_anterior_mean', 'dias_desde_correctivo_anterior_std', 'dias_desde_correctivo_anterior_min', 'dias_desde_correctivo_anterior_max', 'correctivos_previos_max', 'mes', 'dia_semana', 'fin_mes']
Parquet guardado en: C:\Users\josep\Desktop\repo_mantenimiento_predictivo\data\processed\features.parquet


,placa_patente,fecha_evento,dias_desde_correctivo_anterior,correctivos_previos,correctivos_ult_7d,correctivos_ult_5d,correctivos_ult_3d,mes,dia_semana,fin_mes,correctivo_prox_7d,correctivo_prox_5d,correctivo_prox_3d
0,GCBB90,2025-09-29 04:33:00,NaN,0,1.0,1.0,1.0,9,0,1,False,False,False
1,GCBB90,2025-11-02 16:27:24,34.496111,1,1.0,1.0,1.0,11,6,0,False,False,False
2,GCBB90,2025-11-11 21:19:15,9.202674,2,1.0,1.0,1.0,11,1,0,False,False,False
3,GCBB90,2025-12-23 21:01:00,41.987326,3,1.0,1.0,1.0,12,1,0,False,False,False
4,LXWR68,2025-11-03 19:31:55,NaN,0,1.0,1.0,1.0,11,0,0,True,True,True
5,LXWR68,2025-11-06 19:23:35,2.994213,1,2.0,2.0,2.0,11,3,0,False,False,False
6,LXWR68,2025-12-05 14:56:00,28.814178,2,1.0,1.0,1.0,12,4,0,True,True,True
7,LXWR68,2025-12-08 15:31:00,3.024306,3,2.0,2.0,1.0,12,0,0,True,True,True
8,LXWR68,2025-12-09 12:58:00,0.893750,4,3.0,3.0,2.0,12,1,0,True,True,True
9,LXWR68,2025-12-11 21:28:00,2.354167,5,4.0,3.0,2.0,12,3,0,True,True,True
